## **Claude SDK Tutorial**

https://anthropic.skilljar.com/claude-with-the-anthropic-api

---

In [1]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Make a request
def chat(messages: list[object], system_prompt=None, temperature=1.0, stop_sequences=[]) -> str:

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system_prompt:
        params["system"] = system_prompt

    message = client.messages.create(**params) # unpack dict into keyword args
    return message.content[0].text

---

## **Streaming**

In [9]:
# Manaul streaming
messages = []

add_user_message(messages, "Write one sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01HDjhHa2axdJPs5LBiX9CaW', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=6, server_tool_use=None, service_tier='standard'), stop_details=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A fake database is a sim', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ulated or mock data storage system that contains artificially generated, placeholder', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta

In [10]:
# Built-in streaming capability
messages = []

add_user_message(messages, "Write one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")
        # pass

# Collect all events e.g., For record keeping in DB
# stream.get_final_message()

A fake database is a simulated or mock data repository that contains fabricated records designed to mimic real data structures and relationships for testing, development, or demonstration purposes without exposing actual sensitive information.

---

## **Structured Data Using Prefill Assistant Message & Stop-Sequences**

In [11]:
messages = []

add_user_message(messages, "Generate very short event bridge rule as json")

# Prefill assistant message
add_assistant_message(messages, "```json") # writes out response after the text containing "json"

# Stop-sequence
text = chat(messages, stop_sequences=["```"])

text

'\n{\n  "Name": "my-rule",\n  "EventPattern": {\n    "source": ["my.app"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [12]:
import json

json.loads(text.strip())

{'Name': 'my-rule',
 'EventPattern': {'source': ['my.app'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [13]:
from tracemalloc import stop

messages = []

add_user_message(messages, "Generate three different sample AWS CLI commands. Each should be very short")

add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

# Stop-sequence
text = chat(messages, stop_sequences=["```"])

text

'\naws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users\n'